In [1]:
import pypsa
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import geopandas as gpd
from pypsa.plot import add_legend_lines, add_legend_patches, add_legend_semicircles
import yaml
from pathlib import Path
import pandas as pd
import yaml


Set parameter TokenServer to value "sophia1.hpc.ait.dtu.dk"


**Set Up**

In [2]:
fn = 'resources/Iberic5_test/networks/base_s_5__12h_2050.nc'


In [3]:
n= pypsa.Network(fn)

config = yaml.safe_load(Path("config/config.iberic5.yaml").read_text())


INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores, sub_networks


In [4]:
p = Path(fn)  
try:
   if p.exists():
       p.unlink()
       print(f"Deleted {p}")
   else:
       print(f"File not found: {p}")
except Exception as e:
    print(f"Failed to delete {p}: {e}")

Deleted resources/Iberic5_test/networks/base_s_5__12h_2050.nc


**Options**

In [5]:
ongrid=False
cluster_cost_reduction=1
cluster_size=1000   
renewables={"solar",'solar-hsat','onwind'}

In [6]:
nodes_with_clusters = n.buses.loc[
    n.buses.index.str[:2].isin(config['countries']) &
    (n.buses['carrier'] == 'AC')
].index.tolist()




In [7]:
nodes_with_clusters

['ES0 0', 'ES1 0', 'FR0 0', 'FR0 1', 'PT0 0']

**Buses and Generators of the Cluster Addition**

In [8]:
def assign_cluster_generators_and_electricity_buses(n, config, cluster_size, cluster_cost_reduction, renewables):
    
    nodes_renewables_cf = {}                #dictionary of dataframes by node and renewable type, sorting the generators by average capacity factor (descending order)
    clusters_generators={}                      #dictionary of dataframes by node and renewable type, containing the generators assigned to the cluster  

    for node in nodes_with_clusters:
        for renewable in renewables:

            nodes_renewables_cf[(node, renewable)] = pd.DataFrame(
                index=n.generators['p_nom_max'].loc[n.generators.index.astype(str).str.contains(rf"{node}.*{renewable}$")].index,
                columns=["p_max_pu","p_nom_max"]  
            )
            print(nodes_renewables_cf[(node, renewable)])

            clusters_generators[(node, renewable)] = pd.DataFrame()

            #we are considering the highest mean p_min_pu to determine the best generators per renewable available

            nodes_renewables_cf[(node, renewable)] ["p_max_pu"] = n.generators_t['p_max_pu'].loc[:, n.generators_t['p_max_pu'].columns.astype(str).str.contains(rf"{node}.*{renewable}$")].mean()
            nodes_renewables_cf[(node, renewable)] ["p_nom_max"] = n.generators['p_nom_max'].loc[n.generators.index.astype(str).str.contains(rf"{node}.*{renewable}$")]

            nodes_renewables_cf[(node, renewable)] = nodes_renewables_cf[(node, renewable)].sort_values("p_max_pu", ascending=False)

            print(nodes_renewables_cf[(node, renewable)])

            #print(nodes_renewables_cf[(country, renewable)])

            number_gen=0



            while  nodes_renewables_cf[(node, renewable)].iloc[0:number_gen+1]["p_nom_max"].sum() <= cluster_size:

                if number_gen > len(nodes_renewables_cf[(node, renewable)]):
                    raise ValueError(f"Not enough {renewable} generators to reach cluster_size.")
                
                number_gen=number_gen+1

            #print(f"{renewable} generators in cluster: {number_gen+1}")

            clusters_generators[(node, renewable)]  = n.generators.loc[nodes_renewables_cf[(node, renewable)].index[0:number_gen+1]]
            remaining_capacity = nodes_renewables_cf[(node, renewable)].iloc[0:number_gen+1]["p_nom_max"].sum() - cluster_size
            #nodes_renewables_cf[(country, renewable)].iloc[number_gen]["p_nom_max"] = remaining_capacity maybe it is better to do this step later

            print(f"Remaining top {renewable} capacity outside the cluster: {remaining_capacity} MW")

            
            clusters_generators[(node, renewable)].loc[clusters_generators[(node, renewable)].index[number_gen], "p_nom_max"] = cluster_size - clusters_generators[(node, renewable)].loc[clusters_generators[(node, renewable)].index[0:number_gen],"p_nom_max"].sum()

            print(f"Capacity of the last {renewable} generator adjusted to fit cluster size: {clusters_generators[(node, renewable)].loc[clusters_generators[(node, renewable)].index[number_gen], 'p_nom_max']} MW")

            print(clusters_generators[(node, renewable)])

            for idx in clusters_generators[(node, renewable)].index:

                ### Electricity bus and generators ###

                if not n.buses.index.str.contains(rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' cluster'}$").any():
        
                    n.add(
                        "Bus",
                        name=clusters_generators[(node, renewable)].loc[idx].bus + " cluster",
                        v_nom=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "v_nom"],
                        x=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "x"],
                        y=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "y"],
                        unit=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "unit"],
                        location=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "location"],
                        country=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "country"],
                        carrier=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "carrier"],
                        control=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "control"],
                        substation_lv=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "substation_lv"],
                        substation_off=n.buses.at[clusters_generators[(node, renewable)].loc[idx].bus, "substation_off"],
                    )

                n.add(
                    "Generator",
                    name=clusters_generators[(node, renewable)].loc[idx].name + " cluster",
                    bus=clusters_generators[(node, renewable)].loc[idx].bus + " cluster",
                    carrier=clusters_generators[(node, renewable)].loc[idx].carrier,
                    p_nom_max=clusters_generators[(node, renewable)].loc[idx].p_nom_max,
                    p_max_pu=clusters_generators[(node, renewable)].loc[idx].p_max_pu,
                    marginal_cost=clusters_generators[(node, renewable)].loc[idx].marginal_cost*(1-cluster_cost_reduction),
                    capital_cost=clusters_generators[(node, renewable)].loc[idx].capital_cost*(1-cluster_cost_reduction),
                    efficiency=clusters_generators[(node, renewable)].loc[idx].efficiency,
                    location=clusters_generators[(node, renewable)].loc[idx].location,
                    unit=clusters_generators[(node, renewable)].loc[idx].unit,
                    p_nom_extendable=True,
                    overwrite=True,)


                
                
                n.generators_t['p_max_pu'][clusters_generators[(node, renewable)].loc[idx].name + " cluster"] = n.generators_t['p_max_pu'][clusters_generators[(node, renewable)].loc[idx].name]


                ### H2 bus ##

                if not n.buses.index.str.contains(rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2 cluster'}$").any():

                    n.add(
                        "Bus",
                        name=clusters_generators[(node, renewable)].loc[idx].bus + " H2 cluster",
                        v_nom=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "v_nom"],
                        x=n.buses.at[rf"{clusters_generators[(node,renewable)].loc[idx].bus + ' H2'}", "x"],
                        y=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "y"],
                        unit=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "unit"],
                        location=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "location"],
                        country=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "country"],
                        carrier=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "carrier"],
                        control=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "control"],
                        substation_lv=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "substation_lv"],
                        substation_off=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' H2'}", "substation_off"],
                    )

                ### methanol bus ###

                if not n.buses.index.str.contains(rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' methanol cluster'}$").any():

                    n.add(
                        "Bus",
                        name=clusters_generators[(node, renewable)].loc[idx].bus + " methanol cluster",
                        v_nom=n.buses.at["EU methanol", "v_nom"],
                        x=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus}", "x"],
                        y=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus }", "y"],
                        unit=n.buses.at["EU methanol", "unit"],
                        location=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus  }", "location"],
                        country=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus }", "country"],
                        carrier=n.buses.at["EU methanol", "carrier"],
                        control=n.buses.at["EU methanol", "control"],
                        substation_lv=n.buses.at["EU methanol", "substation_lv"],
                        substation_off=n.buses.at["EU methanol", "substation_off"],
                    )
                
                ### Batteries bus ###

                if not n.buses.index.str.contains(rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery cluster'}$").any():

                    n.add(
                        "Bus",
                        name=clusters_generators[(node, renewable)].loc[idx].bus + " battery cluster",
                        v_nom=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "v_nom"],
                        x=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "x"],
                        y=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "y"],
                        unit=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "unit"],
                        location=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "location"],
                        country=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "country"],
                        carrier=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "carrier"],
                        control=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "control"],
                        substation_lv=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "substation_lv"],
                        substation_off=n.buses.at[rf"{clusters_generators[(node, renewable)].loc[idx].bus + ' battery'}", "substation_off"],
                    )



                if idx == nodes_renewables_cf[(node, renewable)].iloc[number_gen].name:
                    n.generators.loc[n.generators.index == idx, "p_nom_max"] = nodes_renewables_cf[(node, renewable)].iloc[0:number_gen+1]["p_nom_max"].sum() - cluster_size

                    print(f"Residual capacity of generator {clusters_generators[(node, renewable)].loc[idx].name} is {n.generators.loc[n.generators.index == idx, 'p_nom_max']} MW")
                
                else:


                    n.remove(
                            "Generator",
                            name=clusters_generators[(node, renewable)].loc[idx].name,
                    )



            

    return n

n= assign_cluster_generators_and_electricity_buses(n, config, cluster_size, cluster_cost_reduction, renewables)
            



            

            





        




              p_max_pu p_nom_max
name                            
ES0 0 0 solar      NaN       NaN
ES0 0 1 solar      NaN       NaN
ES0 0 2 solar      NaN       NaN
ES0 0 3 solar      NaN       NaN
ES0 0 4 solar      NaN       NaN
               p_max_pu      p_nom_max
name                                  
ES0 0 4 solar  0.184273  459237.019329
ES0 0 3 solar  0.173990  445414.825842
ES0 0 2 solar  0.159308   89004.159684
ES0 0 1 solar  0.141897   55237.238064
ES0 0 0 solar  0.126011   35334.976939
Remaining top solar capacity outside the cluster: 458237.0193292077 MW
Capacity of the last solar generator adjusted to fit cluster size: 1000.0 MW
                 bus control type    p_nom  p_nom_mod  p_nom_extendable  \
name                                                                      
ES0 0 4 solar  ES0 0      PQ       21077.1        0.0              True   

               p_nom_min  p_nom_max  p_nom_set  p_min_pu  ...  up_time_before  \
name                                     

                    p_max_pu     p_nom_max
name                                      
ES1 0 0 solar-hsat  0.206231  10404.431317
Remaining top solar-hsat capacity outside the cluster: 9404.431317480601 MW
Capacity of the last solar-hsat generator adjusted to fit cluster size: 1000.0 MW
                      bus control type  p_nom  p_nom_mod  p_nom_extendable  \
name                                                                         
ES1 0 0 solar-hsat  ES1 0      PQ         0.0        0.0              True   

                    p_nom_min  p_nom_max  p_nom_set  p_min_pu  ...  \
name                                                           ...   
ES1 0 0 solar-hsat        0.0     1000.0        NaN       0.0  ...   

                    up_time_before  down_time_before  ramp_limit_up  \
name                                                                  
ES1 0 0 solar-hsat               1                 0            NaN   

                    ramp_limit_down  ramp_limit_start

Residual capacity of generator FR0 0 4 onwind is name
FR0 0 4 onwind    3232.644081
Name: p_nom_max, dtype: float64 MW
              p_max_pu p_nom_max
name                            
FR0 1 0 solar      NaN       NaN
FR0 1 1 solar      NaN       NaN
FR0 1 2 solar      NaN       NaN
FR0 1 3 solar      NaN       NaN
FR0 1 4 solar      NaN       NaN
               p_max_pu      p_nom_max
name                                  
FR0 1 4 solar  0.166525   49568.068712
FR0 1 3 solar  0.156945   78613.042096
FR0 1 2 solar  0.139043   98260.848956
FR0 1 1 solar  0.124760  195904.871787
FR0 1 0 solar  0.115068  180693.467677
Remaining top solar capacity outside the cluster: 48568.068711796455 MW
Capacity of the last solar generator adjusted to fit cluster size: 1000.0 MW
                 bus control type   p_nom  p_nom_mod  p_nom_extendable  \
name                                                                     
FR0 1 4 solar  FR0 1      PQ       1854.6        0.0              True   

     

Residual capacity of generator PT0 0 4 solar is name
PT0 0 4 solar    4289.374999
Name: p_nom_max, dtype: float64 MW
                   p_max_pu p_nom_max
name                                 
PT0 0 0 solar-hsat      NaN       NaN
                    p_max_pu      p_nom_max
name                                       
PT0 0 0 solar-hsat  0.211594  119233.277003
Remaining top solar-hsat capacity outside the cluster: 118233.27700270024 MW
Capacity of the last solar-hsat generator adjusted to fit cluster size: 1000.0 MW
                      bus control type  p_nom  p_nom_mod  p_nom_extendable  \
name                                                                         
PT0 0 0 solar-hsat  PT0 0      PQ         0.0        0.0              True   

                    p_nom_min  p_nom_max  p_nom_set  p_min_pu  ...  \
name                                                           ...   
PT0 0 0 solar-hsat        0.0     1000.0        NaN       0.0  ...   

                    up_time_befo

In [9]:
n.buses.loc[n.buses.index.str.contains("methanol cluster")]

,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network,country,substation_lv,substation_off
name,,,,,,,,,,,,,,,,
ES0 0 methanol cluster,1.0,,-3.384744,40.676192,methanol,MWh_th,ES0 0,1.0,0.0,inf,PQ,,,ES,NaN,NaN
ES1 0 methanol cluster,1.0,,2.704258,39.609088,methanol,MWh_th,ES1 0,1.0,0.0,inf,PQ,,,ES,NaN,NaN
FR0 0 methanol cluster,1.0,,1.259022,48.118048,methanol,MWh_th,FR0 0,1.0,0.0,inf,PQ,,,FR,NaN,NaN
FR0 1 methanol cluster,1.0,,5.298513,45.814195,methanol,MWh_th,FR0 1,1.0,0.0,inf,PQ,,,FR,NaN,NaN
PT0 0 methanol cluster,1.0,,-8.248102,40.187334,methanol,MWh_th,PT0 0,1.0,0.0,inf,PQ,,,PT,NaN,NaN


In [10]:
n.generators_t['p_max_pu']

name,ES0 0 0 offwind-ac,ES0 0 0 offwind-float,ES0 0 0 onwind,ES0 0 0 solar,ES0 0 0 solar rooftop,ES0 0 0 solar-hsat,ES0 0 1 onwind,ES0 0 1 solar,ES0 0 1 solar rooftop,ES0 0 2 onwind,...,FR0 0 0 solar-hsat cluster,FR0 0 4 onwind cluster,FR0 1 4 solar cluster,FR0 1 0 solar-hsat cluster,FR0 1 4 onwind cluster,FR0 1 3 onwind cluster,PT0 0 4 solar cluster,PT0 0 0 solar-hsat cluster,PT0 0 4 onwind cluster,PT0 0 3 onwind cluster
snapshot,,,,,,,,,,,,,,,,,,,,,
2013-01-01 00:00:00,0.342254,0.333113,0.049318,0.037477,0.037477,0.093278,0.192643,0.055660,0.055660,0.180918,...,0.072534,0.847802,0.042297,0.033009,0.408937,0.171382,0.112810,0.090866,0.342515,0.258693
2013-01-01 12:00:00,0.219415,0.147410,0.059962,0.038434,0.038434,0.110425,0.176467,0.073018,0.073018,0.330759,...,0.064181,0.749849,0.033328,0.029348,0.227458,0.360703,0.180491,0.130074,0.244952,0.205997
2013-01-02 00:00:00,0.290413,0.161220,0.059708,0.036256,0.036256,0.146940,0.214296,0.070821,0.070821,0.409495,...,0.079434,0.318092,0.144370,0.087414,0.915105,0.743912,0.119291,0.138024,0.237014,0.191929
2013-01-02 12:00:00,0.207759,0.146617,0.092614,0.061338,0.061338,0.149635,0.348377,0.110123,0.110123,0.458676,...,0.058988,0.516327,0.126516,0.069211,0.995496,0.851249,0.211708,0.176557,0.118110,0.088451
2013-01-03 00:00:00,0.352395,0.261316,0.149679,0.128921,0.128921,0.177738,0.189741,0.126977,0.126977,0.192161,...,0.030890,0.418081,0.164235,0.097425,0.813266,0.497093,0.139424,0.160058,0.179030,0.184967
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2013-12-29 12:00:00,0.335162,0.237284,0.022912,0.108095,0.108095,0.169477,0.125644,0.109019,0.109019,0.428298,...,0.082062,0.696327,0.132989,0.058290,0.837285,0.635113,0.222785,0.168096,0.040636,0.016816
2013-12-30 00:00:00,0.341877,0.271109,0.031853,0.049115,0.049115,0.144615,0.128578,0.047919,0.047919,0.478825,...,0.047679,0.991819,0.169136,0.134691,0.538704,0.239848,0.123433,0.085930,0.526202,0.314559
2013-12-30 12:00:00,0.347182,0.339090,0.038695,0.051238,0.051238,0.128317,0.151031,0.053414,0.053414,0.461388,...,0.036821,0.657245,0.135622,0.083180,0.007083,0.005135,0.209965,0.101364,0.865340,0.585097


In [11]:
n.generators.loc[n.generators.index.str.contains('cluster')]

,bus,control,type,p_nom,p_nom_mod,p_nom_extendable,p_nom_min,p_nom_max,p_nom_set,p_min_pu,...,up_time_before,down_time_before,ramp_limit_up,ramp_limit_down,ramp_limit_start_up,ramp_limit_shut_down,weight,p_nom_opt,location,unit
name,,,,,,,,,,,,,,,,,,,,,
ES0 0 4 solar cluster,ES0 0 cluster,PQ,,0.0,0.0,True,0.0,1000.000000,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
ES0 0 0 solar-hsat cluster,ES0 0 cluster,PQ,,0.0,0.0,True,0.0,1000.000000,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
ES0 0 4 onwind cluster,ES0 0 cluster,PQ,,0.0,0.0,True,0.0,1000.000000,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
ES1 0 3 solar cluster,ES1 0 cluster,PQ,,0.0,0.0,True,0.0,1000.000000,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
ES1 0 0 solar-hsat cluster,ES1 0 cluster,PQ,,0.0,0.0,True,0.0,1000.000000,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
ES1 0 4 onwind cluster,ES1 0 cluster,PQ,,0.0,0.0,True,0.0,789.903930,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
ES1 0 2 onwind cluster,ES1 0 cluster,PQ,,0.0,0.0,True,0.0,210.096070,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
FR0 0 4 solar cluster,FR0 0 cluster,PQ,,0.0,0.0,True,0.0,147.862118,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,
FR0 0 2 solar cluster,FR0 0 cluster,PQ,,0.0,0.0,True,0.0,852.137882,NaN,0.0,...,1,0,NaN,NaN,1.0,1.0,1.0,0.0,,


**Links of the Cluster Addition**

In [12]:
def add_cluster_links(n, nodes_with_clusters, cluster_cost_reduction, ongrid):

    for node in nodes_with_clusters:

        ### H2 Electrolysis ###

        link_name = f"{node} H2 Electrolysis"

        n.add(
            "Link",
            name=link_name + " cluster",
            bus0=n.links.at[link_name, "bus0"] + " cluster",
            bus1=n.links.at[link_name, "bus1"] + " cluster",
            p_nom_extendable=n.links.at[link_name, "p_nom_extendable"],
            carrier=n.links.at[link_name, "carrier"],
            efficiency=n.links.at[link_name, "efficiency"],
            capital_cost=n.links.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.links.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            lifetime=n.links.at[link_name, "lifetime"],
            reversed=False,
            overwrite=True,
        )

        ### Methanolization ###

        ### Methanolization ###

        link_name = f"{node} methanolisation"
        cluster_methanol_bus_name = f"{node} methanol cluster"

        n.add(
            "Link",
            name=link_name + " cluster",
            bus0=n.links.at[link_name, "bus0"] + " cluster",
            bus1=cluster_methanol_bus_name,
            bus2=n.links.at[link_name, "bus2"] + " cluster",
            bus3=n.links.at[link_name, "bus3"],
            bus4=n.links.at[link_name, "bus4"],
            p_nom_extendable=n.links.at[link_name, "p_nom_extendable"],
            p_min_pu=n.links.at[link_name, "p_min_pu"],
            carrier=n.links.at[link_name, "carrier"],
            efficiency=n.links.at[link_name, "efficiency"],
            efficiency2=n.links.at[link_name, "efficiency2"],
            efficiency3=n.links.at[link_name, "efficiency3"],
            efficiency4=n.links.at[link_name, "efficiency4"],
            capital_cost=n.links.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.links.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            lifetime=n.links.at[link_name, "lifetime"],
            reversed=False,
            overwrite=True,
        )

        n.add(
            "Link",
            name=f"{node} methanol cluster",
            bus0=cluster_methanol_bus_name,
            bus1=n.links.at[link_name, "bus1"],
            p_nom_extendable=True,
            carrier=n.buses.loc[n.links.at[link_name, "bus1"], "carrier"],  
            efficiency=1.0,
            capital_cost=0.0,
            marginal_cost=0.0,
            reversed=False,
            overwrite=True,


        )

    if ongrid==True :

        ### Electricity connection to grid ###

        link_name = f"{node} electricity cluster"
        
        n.add(
            "Link",
            name=link_name,
            bus0=f"{node} cluster",
            bus1=f"{node}",
            carrier=n.buses.at[f"{node}", "carrier"],  
            p_nom_extendable=True,
            efficiency=1.0,
            capital_cost=0.0,
            marginal_cost=0.0,
            reversed=False,
            overwrite=True,
        )

        link_name = f"{node} electricity cluster back"
        n.add(
            "Link",
            name=link_name,
            bus0=f"{node}",
            bus1=f"{node} cluster",
            carrier=n.buses.at[f"{node}", "carrier"],  
            p_nom_extendable=True,
            efficiency=1.0,
            capital_cost=0.0,
            marginal_cost=0.0,
            reversed=True,
            overwrite=True,
        )

    else:
        if f"{node} cluster electricity" in n.links.index:
            n.remove(
                "Link",
                name=f"{node} cluster electricity",
            )
        if f"{node} cluster electricity back" in n.links.index:
            n.remove(
                "Link",
                name=f"{node} cluster electricity back",
            )

    return n

n = add_cluster_links(n, nodes_with_clusters, cluster_cost_reduction, ongrid)



        


        

In [13]:
n.links.loc[n.links.index.str.contains("methanolisation cluster")]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
name,,,,,,,,,,,,,,,,,,,,,
ES0 0 methanolisation cluster,ES0 0 H2 cluster,ES0 0 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
ES1 0 methanolisation cluster,ES1 0 H2 cluster,ES1 0 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
FR0 0 methanolisation cluster,FR0 0 H2 cluster,FR0 0 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
FR0 1 methanolisation cluster,FR0 1 H2 cluster,FR0 1 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
PT0 0 methanolisation cluster,PT0 0 H2 cluster,PT0 0 methanol cluster,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


**Storages of the Cluster Addition**

In [14]:
def add_cluster_storages(n, nodes_with_clusters, cluster_cost_reduction):

    for node in nodes_with_clusters:

        link_name = f"{node} H2 Store"

    
        n.add("Store",
            name=link_name + " cluster",
            bus=n.stores.at[link_name, "bus"] + " cluster",
            carrier=n.stores.at[link_name, "carrier"],
            e_nom_extendable=True,
            capital_cost=n.stores.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.stores.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            e_initial_per_period=n.stores.at[link_name, "e_initial_per_period"],
            e_cyclic=n.stores.at[link_name, "e_cyclic"],
            e_cyclic_per_period=n.stores.at[link_name, "e_cyclic_per_period"],
            overwrite=True,
            )
        
        link_name = f"{node} battery"


        n.add(
                "Link",
                name=link_name + " charger cluster",
                bus0=f"{node} cluster",
                bus1=f"{node} battery cluster",
                carrier=n.buses.at[link_name, "carrier"],   
                p_nom_extendable=True,
                efficiency=1.0,
                capital_cost=0.0,
                marginal_cost=0.0,
                reversed=False,
                overwrite=True,
            )
        n.add(
                "Link",
                name=link_name + " discharger cluster",
                bus0=f"{node} battery cluster",
                bus1=f"{node} cluster",
                carrier=n.buses.at[link_name, "carrier"],
                p_nom_extendable=True,
                efficiency=1.0,
                capital_cost=0.0,
                marginal_cost=0.0,
                reversed=True,
                overwrite=True,
            )

        n.add("Store",
            name=link_name + " cluster" ,
            bus=f"{node} battery cluster",
            carrier=n.stores.at[link_name, "carrier"],
            e_nom_extendable=True,
            capital_cost=n.stores.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
            marginal_cost=n.stores.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
            e_initial_per_period=n.stores.at[link_name, "e_initial_per_period"],
            e_cyclic=n.stores.at[link_name, "e_cyclic"],
            e_cyclic_per_period=n.stores.at[link_name, "e_cyclic_per_period"],
            overwrite=True,
            )

        # link_name= f"{node} co2 stored"

        # n.add("Store",
        #     name=link_name + " cluster" ,
        #     bus=f"{node} co2 stored cluster",
        #     carrier=n.stores.at[link_name, "carrier"],
        #     e_nom_extendable=True,
        #     capital_cost=n.stores.at[link_name, "capital_cost"]*(1-cluster_cost_reduction),
        #     marginal_cost=n.stores.at[link_name, "marginal_cost"]*(1-cluster_cost_reduction),
        #     e_initial_per_period=n.stores.at[link_name, "e_initial_per_period"],
        #     e_cyclic=n.stores.at[link_name, "e_cyclic"],
        #     e_cyclic_per_period=n.stores.at[link_name, "e_cyclic_per_period"],
        #     overwrite=True,
        #     )
        
    return n

n = add_cluster_storages(n, nodes_with_clusters, cluster_cost_reduction)





In [15]:
n.links["reversed"] = n.links["reversed"].fillna(False).astype(bool)


**Printing to Check**

In [16]:
n.links.loc[n.links["bus1"].str.contains('methanol')]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
name,,,,,,,,,,,,,,,,,,,,,
ES0 0 solid biomass biomass-to-methanol,ES0 0 solid biomass,EU methanol,,biomass-to-methanol,0.6500,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
ES1 0 solid biomass biomass-to-methanol,ES1 0 solid biomass,EU methanol,,biomass-to-methanol,0.6500,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
FR0 0 solid biomass biomass-to-methanol,FR0 0 solid biomass,EU methanol,,biomass-to-methanol,0.6500,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
FR0 1 solid biomass biomass-to-methanol,FR0 1 solid biomass,EU methanol,,biomass-to-methanol,0.6500,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
PT0 0 solid biomass biomass-to-methanol,PT0 0 solid biomass,EU methanol,,biomass-to-methanol,0.6500,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
EU industry methanol,EU methanol,EU industry methanol,,industry methanol,1.0000,True,0,inf,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
ES0 0 methanolisation,ES0 0 H2,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
ES1 0 methanolisation,ES1 0 H2,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
FR0 0 methanolisation,FR0 0 H2,EU methanol,,methanolisation,0.8787,True,0,20.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0


In [17]:
n.links.loc[n.links.index.str.contains("Electrolysis")]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
name,,,,,,,,,,,,,,,,,,,,,
ES0 0 H2 Electrolysis,ES0 0,ES0 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
ES1 0 H2 Electrolysis,ES1 0,ES1 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
FR0 0 H2 Electrolysis,FR0 0,FR0 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
FR0 1 H2 Electrolysis,FR0 1,FR0 1 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
PT0 0 H2 Electrolysis,PT0 0,PT0 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
ES0 0 H2 Electrolysis cluster,ES0 0 cluster,ES0 0 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
ES1 0 H2 Electrolysis cluster,ES1 0 cluster,ES1 0 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
FR0 0 H2 Electrolysis cluster,FR0 0 cluster,FR0 0 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
FR0 1 H2 Electrolysis cluster,FR0 1 cluster,FR0 1 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


In [18]:
n.links.loc[n.links["carrier"].str.contains('H2 Electrolysis')]

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
name,,,,,,,,,,,,,,,,,,,,,
ES0 0 H2 Electrolysis,ES0 0,ES0 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
ES1 0 H2 Electrolysis,ES1 0,ES1 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
FR0 0 H2 Electrolysis,FR0 0,FR0 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
FR0 1 H2 Electrolysis,FR0 1,FR0 1 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
PT0 0 H2 Electrolysis,PT0 0,PT0 0 H2,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
ES0 0 H2 Electrolysis cluster,ES0 0 cluster,ES0 0 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
ES1 0 H2 Electrolysis cluster,ES1 0 cluster,ES1 0 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
FR0 0 H2 Electrolysis cluster,FR0 0 cluster,FR0 0 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
FR0 1 H2 Electrolysis cluster,FR0 1 cluster,FR0 1 H2 cluster,,H2 Electrolysis,0.6994,True,0,25.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN


In [19]:
n.buses.loc[n.buses.index.str.contains("cluster")]

,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network,country,substation_lv,substation_off
name,,,,,,,,,,,,,,,,
ES0 0 cluster,380.0,,-3.384744,40.676192,AC,MWh_el,ES0 0,1.0,0.0,inf,Slack,,,ES,1.0,1.0
ES0 0 H2 cluster,1.0,,-3.384744,40.676192,H2,MWh_LHV,ES0 0,1.0,0.0,inf,PQ,,,ES,NaN,NaN
ES0 0 methanol cluster,1.0,,-3.384744,40.676192,methanol,MWh_th,ES0 0,1.0,0.0,inf,PQ,,,ES,NaN,NaN
ES0 0 battery cluster,1.0,,-3.384744,40.676192,battery,MWh_el,ES0 0,1.0,0.0,inf,PQ,,,ES,NaN,NaN
ES1 0 cluster,380.0,,2.704258,39.609088,AC,MWh_el,ES1 0,1.0,0.0,inf,Slack,,,ES,1.0,1.0
ES1 0 H2 cluster,1.0,,2.704258,39.609088,H2,MWh_LHV,ES1 0,1.0,0.0,inf,PQ,,,ES,NaN,NaN
ES1 0 methanol cluster,1.0,,2.704258,39.609088,methanol,MWh_th,ES1 0,1.0,0.0,inf,PQ,,,ES,NaN,NaN
ES1 0 battery cluster,1.0,,2.704258,39.609088,battery,MWh_el,ES1 0,1.0,0.0,inf,PQ,,,ES,NaN,NaN
FR0 0 cluster,380.0,,1.259022,48.118048,AC,MWh_el,FR0 0,1.0,0.0,inf,PQ,,,FR,1.0,1.0


In [20]:
n.stores.loc[n.stores.index.str.contains("cluster")]



,bus,type,carrier,e_nom,e_nom_mod,e_nom_extendable,e_nom_min,e_nom_max,e_nom_set,e_min_pu,...,marginal_cost,marginal_cost_quadratic,marginal_cost_storage,capital_cost,standing_loss,active,build_year,lifetime,e_nom_opt,location
name,,,,,,,,,,,,,,,,,,,,,
ES0 0 H2 Store cluster,ES0 0 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,0.0,0.0,True,0,inf,0.0,NaN
ES0 0 battery cluster,ES0 0 battery cluster,,battery,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,0.0,0.0,True,0,inf,0.0,NaN
ES1 0 H2 Store cluster,ES1 0 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,0.0,0.0,True,0,inf,0.0,NaN
ES1 0 battery cluster,ES1 0 battery cluster,,battery,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,0.0,0.0,True,0,inf,0.0,NaN
FR0 0 H2 Store cluster,FR0 0 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,0.0,0.0,True,0,inf,0.0,NaN
FR0 0 battery cluster,FR0 0 battery cluster,,battery,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,0.0,0.0,True,0,inf,0.0,NaN
FR0 1 H2 Store cluster,FR0 1 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,0.0,0.0,True,0,inf,0.0,NaN
FR0 1 battery cluster,FR0 1 battery cluster,,battery,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,0.0,0.0,True,0,inf,0.0,NaN
PT0 0 H2 Store cluster,PT0 0 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,0.0,0.0,True,0,inf,0.0,NaN


In [21]:
n.links.loc[n.links["carrier"]=='DC']

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
name,,,,,,,,,,,,,,,,,,,,,
relation/17631956-250-DC,ES0 0,ES1 0,,DC,0.967969,True,0,inf,400.0,0.0,...,0.0,relation/17631956,LINESTRING (-0.2354476169104305 39.64108480222...,1.0,0.98193,,NaN,,False,530.883129
relation/9934065-320-DC,FR0 1,ES0 0,,DC,0.959574,True,0,inf,1000.0,0.0,...,0.0,relation/9934065,LINESTRING (2.8013989585839205 42.730937367741...,1.0,0.00000,,NaN,,False,905.214019
relation/9934066-320-DC,FR0 1,ES0 0,,DC,0.959574,True,0,inf,1000.0,0.0,...,0.0,relation/9934066,LINESTRING (2.8013989585839205 42.730937367741...,1.0,0.00000,,NaN,,False,905.214019
TYNDP2024_16,ES0 0,FR0 0,,DC,0.959566,True,2028,inf,0.0,0.0,...,1.0,"{name:Biscay Gulf, url:https://eepublicdownloa...","LINESTRING (-2.867 43.367, -0.408943 45.074191)",NaN,0.62000,under_construction,NaN,,False,905.589567
relation/17631956-250-DC-reversed,ES1 0,ES0 0,,DC,0.967969,True,0,inf,400.0,0.0,...,0.0,relation/17631956,LINESTRING (-0.2354476169104305 39.64108480222...,1.0,0.98193,,NaN,,True,530.883129
relation/9934065-320-DC-reversed,ES0 0,FR0 1,,DC,0.959574,True,0,inf,1000.0,0.0,...,0.0,relation/9934065,LINESTRING (2.8013989585839205 42.730937367741...,1.0,0.00000,,NaN,,True,905.214019
relation/9934066-320-DC-reversed,ES0 0,FR0 1,,DC,0.959574,True,0,inf,1000.0,0.0,...,0.0,relation/9934066,LINESTRING (2.8013989585839205 42.730937367741...,1.0,0.00000,,NaN,,True,905.214019
TYNDP2024_16-reversed,FR0 0,ES0 0,,DC,0.959566,True,2028,inf,0.0,0.0,...,1.0,"{name:Biscay Gulf, url:https://eepublicdownloa...","LINESTRING (-2.867 43.367, -0.408943 45.074191)",NaN,0.62000,under_construction,NaN,,True,905.589567


In [22]:
n.stores

,bus,type,carrier,e_nom,e_nom_mod,e_nom_extendable,e_nom_min,e_nom_max,e_nom_set,e_min_pu,...,marginal_cost,marginal_cost_quadratic,marginal_cost_storage,capital_cost,standing_loss,active,build_year,lifetime,e_nom_opt,location
name,,,,,,,,,,,,,,,,,,,,,
co2 atmosphere,co2 atmosphere,,co2,0.0,0.0,True,0.0,inf,NaN,-1.0,...,0.0,0.0,0.0,0.000000,0.0,True,0,inf,0.0,
ES0 0 co2 stored,ES0 0 co2 stored,,co2 stored,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,247.607546,0.0,True,0,inf,0.0,
ES1 0 co2 stored,ES1 0 co2 stored,,co2 stored,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,247.607546,0.0,True,0,inf,0.0,
FR0 0 co2 stored,FR0 0 co2 stored,,co2 stored,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,247.607546,0.0,True,0,inf,0.0,
FR0 1 co2 stored,FR0 1 co2 stored,,co2 stored,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,247.607546,0.0,True,0,inf,0.0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FR0 0 battery cluster,FR0 0 battery cluster,,battery,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,0.000000,0.0,True,0,inf,0.0,NaN
FR0 1 H2 Store cluster,FR0 1 H2 cluster,,H2 Store,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,0.000000,0.0,True,0,inf,0.0,NaN
FR0 1 battery cluster,FR0 1 battery cluster,,battery,0.0,0.0,True,0.0,inf,NaN,0.0,...,0.0,0.0,0.0,0.000000,0.0,True,0,inf,0.0,NaN


In [23]:
n.links.loc[n.links["bus0"]=='EU methanol']

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
name,,,,,,,,,,,,,,,,,,,,,
ES0 0 OCGT methanol,EU methanol,ES0 0,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
ES1 0 OCGT methanol,EU methanol,ES1 0,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
FR0 0 OCGT methanol,EU methanol,FR0 0,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
FR0 1 OCGT methanol,EU methanol,FR0 1,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
PT0 0 OCGT methanol,EU methanol,PT0 0,,OCGT methanol,0.43,True,0,25.0,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
EU industry methanol,EU methanol,EU industry methanol,,industry methanol,1.00,True,0,inf,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0
EU shipping methanol,EU methanol,EU shipping methanol,,shipping methanol,1.00,True,0,inf,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.0


In [24]:
n.carriers

,co2_emissions,color,nice_name,max_growth,max_relative_growth
name,,,,,
DC,0.0,#8a1caf,DC,inf,0.0
AC,0.0,#70af1d,AC,inf,0.0
nuclear,0.0,#ff8c00,nuclear,inf,0.0
solar-hsat,0.0,#fdb915,solar-hsat,inf,0.0
offwind-float,0.0,#b5e2fa,Offshore Wind (Floating),inf,0.0
...,...,...,...,...,...
electricity distribution grid,0.0,#97ad8c,electricity distribution grid,inf,0.0
home battery discharger,0.0,#3c5221,home battery discharger,inf,0.0
rural water tanks charger,0.0,#e69487,rural water tanks charger,inf,0.0


In [25]:
n.global_constraints

,type,investment_period,bus,carrier_attribute,sense,constant,mu
name,,,,,,,
lv_limit,transmission_volume_expansion_limit,NaN,,"AC, DC",<=,3.319136e+07,0.0
biomass limit,operational_limit,NaN,,solid biomass,<=,2.092436e+08,0.0
CO2Limit,co2_atmosphere,NaN,,co2_emissions,<=,0.000000e+00,0.0


In [26]:
n.loads

,bus,carrier,type,p_set,q_set,sign,active
name,,,,,,,
ES0 0,ES0 0 low voltage,electricity,,0.0,0.0,-1.0,True
ES1 0,ES1 0 low voltage,electricity,,0.0,0.0,-1.0,True
FR0 0,FR0 0 low voltage,electricity,,0.0,0.0,-1.0,True
FR0 1,FR0 1 low voltage,electricity,,0.0,0.0,-1.0,True
PT0 0,PT0 0 low voltage,electricity,,0.0,0.0,-1.0,True
...,...,...,...,...,...,...,...
FR0 0 urban decentral heat,FR0 0 urban decentral heat,urban decentral heat,,0.0,0.0,-1.0,True
FR0 1 rural heat,FR0 1 rural heat,rural heat,,0.0,0.0,-1.0,True
FR0 1 urban decentral heat,FR0 1 urban decentral heat,urban decentral heat,,0.0,0.0,-1.0,True


In [27]:
n.links

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,under_construction,tags,geometry,dc,underwater_fraction,project_status,energy to power ratio,location,reversed,length_original
name,,,,,,,,,,,,,,,,,,,,,
relation/17631956-250-DC,ES0 0,ES1 0,,DC,0.967969,True,0,inf,400.0,0.0,...,0.0,relation/17631956,LINESTRING (-0.2354476169104305 39.64108480222...,1.0,0.98193,,NaN,,False,530.883129
relation/9934065-320-DC,FR0 1,ES0 0,,DC,0.959574,True,0,inf,1000.0,0.0,...,0.0,relation/9934065,LINESTRING (2.8013989585839205 42.730937367741...,1.0,0.00000,,NaN,,False,905.214019
relation/9934066-320-DC,FR0 1,ES0 0,,DC,0.959574,True,0,inf,1000.0,0.0,...,0.0,relation/9934066,LINESTRING (2.8013989585839205 42.730937367741...,1.0,0.00000,,NaN,,False,905.214019
TYNDP2024_16,ES0 0,FR0 0,,DC,0.959566,True,2028,inf,0.0,0.0,...,1.0,"{name:Biscay Gulf, url:https://eepublicdownloa...","LINESTRING (-2.867 43.367, -0.408943 45.074191)",NaN,0.62000,under_construction,NaN,,False,905.589567
ES0 0 co2 sequestered,ES0 0 co2 stored,ES0 0 co2 sequestered,,co2 sequestered,1.000000,True,0,inf,0.0,0.0,...,NaN,,,NaN,NaN,,NaN,,False,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FR0 0 battery discharger cluster,FR0 0 battery cluster,FR0 0 cluster,,battery,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN
FR0 1 battery charger cluster,FR0 1 cluster,FR0 1 battery cluster,,battery,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN
FR0 1 battery discharger cluster,FR0 1 battery cluster,FR0 1 cluster,,battery,1.000000,True,0,inf,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN


In [28]:
n.links.loc[n.links.index.str.contains("cluster"),n.links.columns.str.contains("bus")]

,bus0,bus1,bus4,bus3,bus2
name,,,,,
ES0 0 H2 Electrolysis cluster,ES0 0 cluster,ES0 0 H2 cluster,,,
ES0 0 methanolisation cluster,ES0 0 H2 cluster,ES0 0 methanol cluster,ES0 0 urban central heat,ES0 0 co2 stored,ES0 0 cluster
ES0 0 methanol cluster,ES0 0 methanol cluster,EU methanol,,,
ES1 0 H2 Electrolysis cluster,ES1 0 cluster,ES1 0 H2 cluster,,,
ES1 0 methanolisation cluster,ES1 0 H2 cluster,ES1 0 methanol cluster,ES1 0 urban central heat,ES1 0 co2 stored,ES1 0 cluster
ES1 0 methanol cluster,ES1 0 methanol cluster,EU methanol,,,
FR0 0 H2 Electrolysis cluster,FR0 0 cluster,FR0 0 H2 cluster,,,
FR0 0 methanolisation cluster,FR0 0 H2 cluster,FR0 0 methanol cluster,FR0 0 urban central heat,FR0 0 co2 stored,FR0 0 cluster
FR0 0 methanol cluster,FR0 0 methanol cluster,EU methanol,,,


In [29]:
print(n.generators.loc[n.generators.index.str.contains("solar")&
    ~n.generators.index.str.contains("solar-hsat") &~n.generators.index.str.contains("solar thermal") &~n.generators.index.str.contains("solar rooftop")])

                                 bus control type    p_nom  p_nom_mod  \
name                                                                    
ES0 0 0 solar                  ES0 0      PQ          51.5        0.0   
ES0 0 1 solar                  ES0 0      PQ         166.1        0.0   
ES0 0 2 solar                  ES0 0      PQ         460.8        0.0   
ES0 0 3 solar                  ES0 0      PQ       14935.0        0.0   
ES0 0 4 solar                  ES0 0      PQ       21077.1        0.0   
ES1 0 0 solar                  ES1 0      PQ          23.4        0.0   
ES1 0 1 solar                  ES1 0      PQ         170.1        0.0   
ES1 0 2 solar                  ES1 0      PQ          47.3        0.0   
ES1 0 3 solar                  ES1 0      PQ          29.9        0.0   
ES1 0 4 solar                  ES1 0      PQ           0.0        0.0   
FR0 0 0 solar                  FR0 0      PQ        1129.6        0.0   
FR0 0 1 solar                  FR0 0      PQ       

In [30]:
print(n.generators.loc[n.generators.index.str.contains("solar cluster"), n.generators.columns.isin(['p_nom_max','carrier','location','pnom_extendable'])])


                         p_nom_max carrier location
name                                               
ES0 0 4 solar cluster  1000.000000   solar         
ES1 0 3 solar cluster  1000.000000   solar         
FR0 0 4 solar cluster   147.862118   solar         
FR0 0 2 solar cluster   852.137882   solar         
FR0 1 4 solar cluster  1000.000000   solar         
PT0 0 4 solar cluster  1000.000000   solar         


In [31]:
n.generators_t['p_max_pu'].loc[:, n.generators_t['p_max_pu'].columns.str.contains("cluster")]

name,ES0 0 4 solar cluster,ES0 0 0 solar-hsat cluster,ES0 0 4 onwind cluster,ES1 0 3 solar cluster,ES1 0 0 solar-hsat cluster,ES1 0 4 onwind cluster,ES1 0 2 onwind cluster,FR0 0 4 solar cluster,FR0 0 2 solar cluster,FR0 0 0 solar-hsat cluster,FR0 0 4 onwind cluster,FR0 1 4 solar cluster,FR0 1 0 solar-hsat cluster,FR0 1 4 onwind cluster,FR0 1 3 onwind cluster,PT0 0 4 solar cluster,PT0 0 0 solar-hsat cluster,PT0 0 4 onwind cluster,PT0 0 3 onwind cluster
snapshot,,,,,,,,,,,,,,,,,,,
2013-01-01 00:00:00,0.083906,0.093278,0.395482,0.100009,0.113146,0.803567,0.580701,0.124987,0.062019,0.072534,0.847802,0.042297,0.033009,0.408937,0.171382,0.112810,0.090866,0.342515,0.258693
2013-01-01 12:00:00,0.127043,0.110425,0.159663,0.126709,0.089815,0.163882,0.103875,0.124410,0.071685,0.064181,0.749849,0.033328,0.029348,0.227458,0.360703,0.180491,0.130074,0.244952,0.205997
2013-01-02 00:00:00,0.141384,0.146940,0.157479,0.101194,0.088808,0.564712,0.203198,0.145279,0.092461,0.079434,0.318092,0.144370,0.087414,0.915105,0.743912,0.119291,0.138024,0.237014,0.191929
2013-01-02 12:00:00,0.185945,0.149635,0.495514,0.056244,0.073223,1.000000,0.367944,0.168262,0.113511,0.058988,0.516327,0.126516,0.069211,0.995496,0.851249,0.211708,0.176557,0.118110,0.088451
2013-01-03 00:00:00,0.154811,0.177738,0.422764,0.060573,0.060424,0.985019,0.833830,0.154826,0.060720,0.030890,0.418081,0.164235,0.097425,0.813266,0.497093,0.139424,0.160058,0.179030,0.184967
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2013-12-29 12:00:00,0.207731,0.169477,0.934417,0.136474,0.114194,0.069487,0.169457,0.105972,0.101738,0.082062,0.696327,0.132989,0.058290,0.837285,0.635113,0.222785,0.168096,0.040636,0.016816
2013-12-30 00:00:00,0.147875,0.144615,1.000000,0.168401,0.189194,0.249386,0.047247,0.109376,0.064809,0.047679,0.991819,0.169136,0.134691,0.538704,0.239848,0.123433,0.085930,0.526202,0.314559
2013-12-30 12:00:00,0.172916,0.128317,0.991790,0.161270,0.141544,0.104872,0.131921,0.120701,0.069692,0.036821,0.657245,0.135622,0.083180,0.007083,0.005135,0.209965,0.101364,0.865340,0.585097


**Exporting**

In [32]:
n.export_to_netcdf(fn)


INFO:pypsa.network.io:Exported network 'Unnamed Network' saved to 'resources/Iberic5_test/networks/base_s_5__12h_2050.nc contains: links, stores, global_constraints, lines, generators, buses, storage_units, sub_networks, carriers, loads


<xarray.Dataset> Size: 1MB
Dimensions:                               (snapshots: 730, links_i: 379,
                                           links_t_efficiency_i: 20,
                                           links_t_p_max_pu_i: 10,
                                           stores_i: 74,
                                           stores_t_e_min_pu_i: 5,
                                           stores_t_e_max_pu_i: 10,
                                           ...
                                           generators_i: 165,
                                           generators_t_p_max_pu_i: 126,
                                           buses_i: 153, storage_units_i: 10,
                                           storage_units_t_inflow_i: 5,
                                           sub_networks_i: 2, carriers_i: 120,
                                           loads_i: 88, loads_t_p_set_i: 25)
Coordinates: (12/18)
  * snapshots                             (snapshots) int64 6kB 0 1 ... 728 729
  * links_i                               (links_i) object 3kB 'relation/1763...
  * links_t_efficiency_i                  (links_t_efficiency_i) object 160B ...
  * links_t_p_max_pu_i                    (links_t_p_max_pu_i) object 80B 'ES...
  * stores_i                              (stores_i) object 592B 'co2 atmosph...
  * stores_t_e_min_pu_i                   (stores_t_e_min_pu_i) object 40B 'E...
    ...                                    ...
  * storage_units_i                       (storage_units_i) object 80B 'ES0 0...
  * storage_units_t_inflow_i              (storage_units_t_inflow_i) object 40B ...
  * sub_networks_i                        (sub_networks_i) object 16B '0' '1'
  * carriers_i                            (carriers_i) object 960B 'DC' ... '...
  * loads_i                               (loads_i) object 704B 'ES0 0' ... '...
  * loads_t_p_set_i                       (loads_t_p_set_i) object 200B 'ES0 ...
Data variables: (12/127)
    snapshots_snapshot                    (snapshots) datetime64[ns] 6kB 2013...
    snapshots_objective                   (snapshots) float64 6kB 12.0 ... 12.0
    snapshots_stores                      (snapshots) float64 6kB 12.0 ... 12.0
    snapshots_generators                  (snapshots) float64 6kB 12.0 ... 12.0
    links_bus0                            (links_i) object 3kB 'ES0 0' ... 'P...
    links_bus1                            (links_i) object 3kB 'ES1 0' ... 'P...
    ...                                    ...
    carriers_color                        (carriers_i) object 960B '#8a1caf' ...
    carriers_nice_name                    (carriers_i) object 960B 'DC' ... '...
    loads_bus                             (loads_i) object 704B 'ES0 0 low vo...
    loads_carrier                         (loads_i) object 704B 'electricity'...
    loads_p_set                           (loads_i) float64 704B 0.0 0.0 ... 0.0
    loads_t_p_set                         (snapshots, loads_t_p_set_i) float64 146kB ...
Attributes:
    network_name:           Unnamed Network
    network_pypsa_version:  1.0.6
    network_srid:           4326
    crs:                    {"_crs": "GEOGCRS[\"WGS 84\",ENSEMBLE[\"World Geo...
    meta:                   {"version": "v2025.07.0", "tutorial": false, "log...